In [1]:
%run training_setup.ipynb

c:\Users\Nourhan Yehia\Desktop\Jupyter\xray_pneumonia_classifier
4172 1044 624
torch.Size([32, 1, 224, 224])
torch.Size([32])
tensor([1, 0, 1, 0, 1, 1, 1, 1, 0, 1])


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [4]:
model = SimpleCNN().to(device)

In [6]:
class_counts = train_df["class"].value_counts()

n_normal = class_counts["NORMAL"]
n_pneu   = class_counts["PNEUMONIA"]

total = n_normal + n_pneu

w_normal = total / (2 * n_normal)
w_pneu   = total / (2 * n_pneu)

class_weights = torch.tensor([w_normal, w_pneu], dtype=torch.float32).to(device)

In [7]:
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [8]:
num_epochs = 5
best_val_loss = float("inf")

for epoch in range(num_epochs):

 
    model.train()
    train_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss /= total
    train_acc = correct / total



    model.eval()
    val_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss /= total
    val_acc = correct / total



    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "models/simple_cnn_best.pth")

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print("-" * 40)

Epoch [1/5]
Train Loss: 0.2965 | Train Acc: 0.8775
Val   Loss: 2.1489 | Val   Acc: 0.2874
----------------------------------------
Epoch [2/5]
Train Loss: 0.2290 | Train Acc: 0.9108
Val   Loss: 1.0402 | Val   Acc: 0.7538
----------------------------------------
Epoch [3/5]
Train Loss: 0.1842 | Train Acc: 0.9276
Val   Loss: 1.4242 | Val   Acc: 0.4693
----------------------------------------
Epoch [4/5]
Train Loss: 0.1559 | Train Acc: 0.9444
Val   Loss: 0.1098 | Val   Acc: 0.9780
----------------------------------------
Epoch [5/5]
Train Loss: 0.1567 | Train Acc: 0.9401
Val   Loss: 1.4339 | Val   Acc: 0.3879
----------------------------------------


In [9]:
model.load_state_dict(torch.load("models/simple_cnn_best.pth"))
model.eval()

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)

In [10]:
correct = 0
total = 0

all_preds = []
all_labels = []

thr = 0.6

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        probs = F.softmax(outputs, dim=1)
        p_pneu = probs[:, 1]

        preds = (p_pneu >= thr).long()

        total += labels.size(0)
        correct += (preds == labels).sum().item()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = correct / total
print(f"Test Accuracy: {test_acc:.4f}")

print(classification_report(all_labels, all_preds, target_names=["NORMAL", "PNEUMONIA"]))

Test Accuracy: 0.8157
              precision    recall  f1-score   support

      NORMAL       0.97      0.53      0.68       234
   PNEUMONIA       0.78      0.99      0.87       390

    accuracy                           0.82       624
   macro avg       0.87      0.76      0.78       624
weighted avg       0.85      0.82      0.80       624



In [11]:
torch.save(model.state_dict(), "resnet18_finall.pth")